# Manual Price Push

Push manual price overrides for specific SKUs using market data or fixed prices.

## How to use
1. Run cells 1-3 (setup + data loading)
2. Edit the `PUSH_LIST` in cell 4 with your product IDs and desired actions
3. Run cell 4 — prices are computed per product × cohort (using cohort-specific market data)
4. Review the output table
5. Set `MODE = 'live'` and run cell 5 to push

> Each product is automatically expanded to all 9 cohorts. Market-based actions use cohort-specific prices.  
> Main/general cohorts (695, 61, 699, 697, 698, 696) are auto-mirrored by the push handler.

## Available Actions
| Action | Description |
|--------|-------------|
| `market_min` | Lowest market price (P0 / minimum) |
| `market_25` | 25th percentile market price |
| `market_50` | Median market price (P50) |
| `market_75` | 75th percentile market price |
| `market_max` | Highest market price (maximum) |
| `market_avg` | Average of min and max market prices |
| `target_margin` | Price from brand-category target margin |
| `step_up` | Next tier above current price in the price list |
| `step_down` | Next tier below current price in the price list |
| `<number>` | Fixed price (e.g. `115`, `23.5`) |

> Only SKUs with stock > 0 are pushed.

In [1]:
# =============================================================================
# SETUP
# =============================================================================
import sys, os
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

sys.path.insert(0, os.path.abspath('.'))
import setup_environment_2
setup_environment_2.initialize_env()

from db import query_snowflake
from constants import WAREHOUSE_MAPPING, COHORT_IDS

os.chdir(os.path.join(os.path.dirname(os.path.abspath('__file__')) or os.getcwd(), 'modules'))
%run push_prices_handler.ipynb
os.chdir('..')

CAIRO_TZ = pytz.timezone('Africa/Cairo')
TODAY = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')
print(f'Ready. Today: {TODAY}')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


Push Prices Handler loaded at 2026-04-26 12:22:48 Cairo time
✓ API credentials loaded successfully
✓ Google Sheets client initialized
Ready. Today: 2026-04-26


In [2]:
# =============================================================================
# LOAD MARKET DATA + WAC + CURRENT PRICES + PACKING UNITS
# =============================================================================
os.chdir('modules')
%run market_data_module_2.ipynb
os.chdir('..')

print('Loading market data (V2)...')
market_data = get_market_data_v2()
print(f'  Market data: {len(market_data)} rows')

print('Loading WAC...')
WAC_QUERY = f'''
SELECT product_id, wac_p
FROM finance.all_cogs c
WHERE wac_p > 0
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) 
        BETWEEN c.from_date AND c.to_date
'''
df_wac = query_snowflake(WAC_QUERY)
print(f'  WAC: {len(df_wac)} rows')

print('Loading current prices...')
PRICES_QUERY = f'''

    SELECT 
        cohort_id,
        pu.product_id,
        pu.packing_unit_id,
        pu.basic_unit_count,
        AVG(cpu.price) AS current_price
    FROM cohort_product_packing_units cpu
    JOIN packing_unit_products pu ON pu.id = cpu.product_packing_unit_id
    WHERE cpu.cohort_id IN ({','.join(str(c) for c in COHORT_IDS)})
        AND cpu.created_at::DATE <> '2023-07-31'
        AND cpu.is_customized = TRUE
        and basic_unit_count = 1 
    GROUP BY 1,2,3,4

'''
df_prices = query_snowflake(PRICES_QUERY)
#df_prices = setup_environment_2.dwh_pg_query(PRICES_QUERY, columns=['cohort_id','product_id','packing_unit_id','basic_unit_count','current_price'])
print(f'  Prices: {len(df_prices)} rows')

print('Loading packing units...')
PU_QUERY = '''
WITH sales_check AS (
    SELECT DISTINCT
        pso.product_id,
        pso.packing_unit_id,
        SUM(pso.total_price) AS nmv
    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    WHERE so.created_at >= CURRENT_DATE - 60 
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
    GROUP BY 1, 2
)
SELECT product_id, packing_unit_id, basic_unit_count
FROM (
    SELECT *, MAX(nmv) OVER (PARTITION BY product_id, is_basic_unit) AS top_nmv
    FROM (
        SELECT 
            pup.product_id,
            pup.packing_unit_id,
            pup.basic_unit_count,
            pup.is_basic_unit,
            COUNT(DISTINCT CASE WHEN pup.basic_unit_count = 1 THEN pup.packing_unit_id END) 
                OVER (PARTITION BY pup.product_id) AS total_basic,
            COALESCE(sc.nmv, 0) AS nmv
        FROM packing_unit_products pup
        LEFT JOIN sales_check sc 
            ON sc.product_id = pup.product_id 
            AND sc.packing_unit_id = pup.packing_unit_id
        WHERE pup.deleted_at IS NULL
    )
)
WHERE nmv = top_nmv OR (top_nmv = 0 AND total_basic <= 1)
'''
df_pus = query_snowflake(PU_QUERY)
print(f'  Packing units: {len(df_pus)} rows')

print('Loading product info...')
PRODUCT_QUERY = '''
SELECT p.id AS product_id,
       CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
       b.name_ar AS brand,
       c.name_ar AS cat
FROM products p
JOIN brands b ON b.id = p.brand_id
JOIN categories c ON c.id = p.category_id
JOIN product_units ON product_units.id = p.unit_id
'''
df_products = query_snowflake(PRODUCT_QUERY)
print(f'  Products: {len(df_products)} rows')

print('Loading target margins...')
MARGIN_QUERY = f'''
SELECT brand, cat, target_margin
FROM (
    SELECT b.name_ar AS brand, c.name_ar AS cat, cplan.margin AS target_margin,
           ROW_NUMBER() OVER (PARTITION BY b.name_ar, c.name_ar ORDER BY cplan.date DESC) AS rn
    FROM performance.commercial_targets cplan
    JOIN brands b ON b.id = cplan.brand_id
    JOIN categories c ON c.id = cplan.category_id
    WHERE cplan.margin IS NOT NULL
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(date) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
)
WHERE rn = 1
'''
df_targets = query_snowflake(MARGIN_QUERY)
print(f'  Target margins: {len(df_targets)} rows')

print('Loading stocks...')
STOCKS_QUERY = '''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT warehouse_id, product_id,
    CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks
FROM (
    SELECT pw.warehouse_id, pw.product_id,
        pw.available_stock::INTEGER AS stocks,
        pw2.available_stock::INTEGER AS stocks_child
    FROM product_warehouse pw
    LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
    LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
    WHERE pw.warehouse_id NOT IN (6, 9, 10)
        AND pw.is_basic_unit = 1
)
'''
df_stocks = query_snowflake(STOCKS_QUERY)
print(f'  Stocks: {len(df_stocks)} rows')

print('\nAll data loaded.')

/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined
get_commercial_min_prices() defined
get_commercial_price_ups() defined
get_margin_boundaries_region() defined
get_margin_boundaries_global() defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()

In [3]:
# =============================================================================
# BUILD LOOKUP TABLE (product_id → market prices + WAC + target margin)
# =============================================================================

# V2 market_data returns (product_id, region, price_tiers, wac_p, target_margin, num_sources, market_data_source)
# Expand to cohorts for lookup
df_market_cohorts = expand_to_cohorts(market_data)

# Lookup is per (product_id, cohort_id)
lookup = df_products.merge(df_wac, on='product_id', how='left')

cohort_df = pd.DataFrame({'cohort_id': COHORT_IDS})
lookup = lookup.merge(cohort_df, how='cross')
lookup = lookup.merge(
    df_market_cohorts[['product_id', 'cohort_id', 'price_tiers', 'target_margin']],
    on=['product_id', 'cohort_id'], how='left'
)
lookup['price_tiers'] = lookup['price_tiers'].apply(lambda x: x if isinstance(x, list) else [])

# Fill target_margin for rows without V2 data
lookup = lookup.merge(df_targets, on=['brand', 'cat'], how='left', suffixes=('', '_tgt'))
for col in ['target_bm', 'target_bm_tgt', 'cat_target_margin', 'cat_target_margin_tgt']:
    if col in lookup.columns:
        lookup['target_margin'] = lookup['target_margin'].fillna(lookup[col])
lookup['target_margin'] = lookup['target_margin'].fillna(0.05)
lookup.drop(columns=[c for c in lookup.columns if c.startswith('target_bm') or c.startswith('cat_target')], inplace=True, errors='ignore')

# Derive market percentile prices from price_tiers list
def tiers_to_percentiles(tiers):
    if not tiers or len(tiers) == 0:
        return pd.Series({'market_min': np.nan, 'market_25': np.nan, 'market_50': np.nan,
                          'market_75': np.nan, 'market_max': np.nan, 'market_avg': np.nan})
    arr = np.array(tiers)
    return pd.Series({
        'market_min': arr[0],
        'market_25': np.percentile(arr, 25),
        'market_50': np.percentile(arr, 50),
        'market_75': np.percentile(arr, 75),
        'market_max': arr[-1],
        'market_avg': (arr[0] + arr[-1]) / 2,
    })

lookup = pd.concat([lookup, lookup['price_tiers'].apply(tiers_to_percentiles)], axis=1)

# Merge current base-unit price per (product_id, cohort_id)
base_prices = df_prices[df_prices['basic_unit_count'] == 1][['cohort_id', 'product_id', 'current_price']].drop_duplicates()
lookup = lookup.merge(base_prices, on=['product_id', 'cohort_id'], how='left')

# Merge stocks (sum across warehouses per product for display; warehouse-level for push filtering)
wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])
product_stocks = df_stocks.groupby('product_id')['stocks'].sum().reset_index().rename(columns={'stocks': 'total_stocks'})
lookup = lookup.merge(product_stocks, on='product_id', how='left')
lookup['total_stocks'] = lookup['total_stocks'].fillna(0).astype(int)

print(f'Lookup table: {len(lookup)} rows (product x cohort)')
print(f'  With market data: {(lookup["price_tiers"].apply(len) > 0).sum()}')
print(f'  With WAC: {lookup["wac_p"].notna().sum()}')
print(f'  With target margin: {lookup["target_margin"].notna().sum()}')

Lookup table: 234077 rows (product x cohort)
  With market data: 44549
  With WAC: 76154
  With target margin: 234077


In [ ]:
#lookup[(lookup['total_stocks']>0)].to_excel('manual_data.xlsx')

In [ ]:
#lookup.to_excel('manual_data.xlsx')

In [ ]:
lookup[(lookup['brand']=='ماريو')&(lookup['total_stocks']>0)]

In [8]:
# =============================================================================
# DEFINE YOUR SKU ACTIONS HERE
# =============================================================================
# Format: (product_id, action)
#   action can be: 'market_min', 'market_25', 'market_50', 'market_75',
#                  'market_max', 'market_avg', 'target_margin',
#                  'step_up', 'step_down', or a number (fixed price)
#
# step_up: next tier above current price in the price_tiers list
# step_down: next tier below current price in the price_tiers list
# Each product is automatically expanded to ALL cohorts.
# Market-based actions use the cohort-specific market price.
# Only SKUs with stock > 0 are pushed.

PUSH_LIST = [
    (434,1126,'market_25'),
(88,1123,'market_25'),
(434,702,'market_25'),
(6497,704,'market_25'),
(215,1123,'market_25'),
(6497,704,'market_25'),
(6497,700,'market_25'),
(6496,1124,'market_25'),
(7630,1126,'market_25'),
(6497,1124,'market_25'),
(248,1125,'market_25'),
(436,1124,'market_25'),
(6494,702,'market_25'),
(9833,701,'market_25'),
(624,702,'market_25'),
(8285,703,'market_25'),
(57,1124,'market_25'),
(6158,700,'market_25'),
(1126,702,'market_25'),
(3431,1123,'target_margin'),
(2438,701,'market_25'),
(6158,1126,'market_25'),
(25900,1125,'market_25'),
(12925,703,'target_margin'),
(5646,704,'market_25'),
(7705,701,'market_25'),
(418,702,'market_25'),
(22558,1125,'market_25'),
(11403,702,'market_25'),
(6268,1123,'market_25'),
(19968,1123,'market_25'),
(3994,701,'market_25'),
(8104,704,'market_25'),
(19969,1123,'market_25'),
(9783,1123,'market_25'),
(955,702,'market_25'),
(4224,701,'market_25'),
(3961,701,'market_25')


]


# =============================================================================
# COMPUTE NEW PRICES (per product × cohort)
# =============================================================================
ACTION_TO_COLUMN = {
    'market_min': 'market_min',
    'market_25': 'market_25',
    'market_50': 'market_50',
    'market_75': 'market_75',
    'market_max': 'market_max',
    'market_avg': 'market_avg',
}

results = []
for product_id,cohort_id, action in PUSH_LIST:
    product_rows = lookup[(lookup['product_id'] == product_id)&(lookup['cohort_id'] == cohort_id)] #
    if product_rows.empty:
        results.append({'product_id': product_id, 'cohort_id': '-', 'action': str(action), 'status': 'NOT FOUND'})
        continue

    for _, row in product_rows.iterrows():
        cohort_id = int(row['cohort_id'])
        wac = row.get('wac_p', None)
        sku = row.get('sku', '')
        brand = row.get('brand', '')

        if isinstance(action, (int, float)):
            base_price = float(action)
            action_label = f'fixed_{action}'
        elif action == 'target_margin':
            tm = row.get('target_margin', 0.05)
            if pd.isna(wac) or wac <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO WAC'})
                continue
            base_price = wac / (1 - tm)
            action_label = f'target_margin ({tm:.1%})'
        elif action == 'step_up':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_up = [t for t in tiers if t > cur + 0.25]
            if not next_up:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT TOP'})
                continue
            base_price = next_up[0]
            action_label = f'step_up ({cur:.2f} -> {base_price:.2f})'
        elif action == 'step_down':
            tiers = row.get('price_tiers', [])
            cur = row.get('current_price', 0)
            if not tiers or pd.isna(cur) or cur <= 0:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO TIERS OR PRICE'})
                continue
            next_down = [t for t in reversed(tiers) if t < cur - 0.5]
            if not next_down:
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'ALREADY AT BOTTOM'})
                continue
            base_price = next_down[0]
            action_label = f'step_down ({cur:.2f} -> {base_price:.2f})'
        elif action in ACTION_TO_COLUMN:
            col = ACTION_TO_COLUMN[action]
            val = row.get(col, None)
            if pd.isna(val):
                results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': action, 'status': 'NO MARKET DATA'})
                continue
            base_price = val
            action_label = action
        else:
            results.append({'product_id': product_id, 'cohort_id': cohort_id, 'sku': sku, 'action': str(action), 'status': 'INVALID ACTION'})
            continue

        base_price_rounded = np.round(base_price * 4) / 4
        margin = (base_price_rounded - wac) / base_price_rounded if (wac and wac > 0 and base_price_rounded > 0) else None

        cur_price = row.get('current_price', None)

        results.append({
            'product_id': product_id,
            'cohort_id': cohort_id,
            'sku': sku,
            'brand': brand,
            'action': action_label,
            'wac': wac,
            'current_price': cur_price,
            'base_price': base_price,
            'new_price': base_price_rounded,
            'margin': margin,
            'status': 'OK',
        })

df_results = pd.DataFrame(results)

ok_count = (df_results['status'] == 'OK').sum()
err_count = len(df_results) - ok_count
products_ok = df_results[df_results['status'] == 'OK']['product_id'].nunique()
print(f'Results: {ok_count} OK across {products_ok} products × cohorts, {err_count} errors/skipped')
if err_count > 0:
    print('\nErrors/Skipped:')
    print(df_results[df_results['status'] != 'OK'][['product_id', 'cohort_id', 'sku', 'action', 'status']].to_string(index=False))
df_results[df_results['status'] == 'OK']

Results: 38 OK across 33 products × cohorts, 0 errors/skipped


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status
0,434,1126,بيبسى كانز جيب - 240 مل,بيبسي,market_25,291.735940,298.00,301.500000,301.50,0.032385,OK
1,88,1123,شاى ليبتون ناعم - 250 جم,ليبتون,market_25,48.615451,51.50,51.625000,51.50,0.056011,OK
2,434,702,بيبسى كانز جيب - 240 مل,بيبسي,market_25,291.735940,298.00,300.500000,300.50,0.029165,OK
3,6497,704,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,869.50,934.437500,934.50,0.021084,OK
4,215,1123,بن عبد المعبود محوج فاتح - 100 جم,بن عبد المعبود,market_25,714.885889,646.50,767.812500,767.75,0.068856,OK
5,6497,704,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,869.50,934.437500,934.50,0.021084,OK
6,6497,700,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,905.00,926.000000,926.00,0.012098,OK
7,6496,1124,سندة زيت خليط - 900 مل,سندة,market_25,815.694978,809.75,878.250000,878.25,0.071227,OK
8,7630,1126,فيورى جولد - 400 مل,فيوري,market_25,171.870248,234.00,221.125000,221.00,0.222307,OK
9,6497,1124,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,904.50,934.437500,934.50,0.021084,OK


In [9]:
#df_results=df_results[df_results['current_price']>df_results['new_price']]
df_results['change'] = (df_results['new_price'] - df_results['current_price'])/df_results['current_price']
print(df_results['margin'].mean())
print(df_results['margin'].median())
print(df_results['margin'].min())
print(df_results['margin'].max())
#df_results=df_results[~df_results['brand'].isin(['كوكا كولا','شويبس'])]
df_results

0.04594666666408776
0.04126486192548594
0.0029723518989641057
0.2223065723523898


,product_id,cohort_id,sku,brand,action,wac,current_price,base_price,new_price,margin,status,change
0,434,1126,بيبسى كانز جيب - 240 مل,بيبسي,market_25,291.735940,298.00,301.500000,301.50,0.032385,OK,0.011745
1,88,1123,شاى ليبتون ناعم - 250 جم,ليبتون,market_25,48.615451,51.50,51.625000,51.50,0.056011,OK,0.000000
2,434,702,بيبسى كانز جيب - 240 مل,بيبسي,market_25,291.735940,298.00,300.500000,300.50,0.029165,OK,0.008389
3,6497,704,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,869.50,934.437500,934.50,0.021084,OK,0.074756
4,215,1123,بن عبد المعبود محوج فاتح - 100 جم,بن عبد المعبود,market_25,714.885889,646.50,767.812500,767.75,0.068856,OK,0.187548
5,6497,704,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,869.50,934.437500,934.50,0.021084,OK,0.074756
6,6497,700,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,905.00,926.000000,926.00,0.012098,OK,0.023204
7,6496,1124,سندة زيت خليط - 900 مل,سندة,market_25,815.694978,809.75,878.250000,878.25,0.071227,OK,0.084594
8,7630,1126,فيورى جولد - 400 مل,فيوري,market_25,171.870248,234.00,221.125000,221.00,0.222307,OK,-0.055556
9,6497,1124,سندة زيت خليط - 1 لتر,سندة,market_25,914.797010,904.50,934.437500,934.50,0.021084,OK,0.033167


In [ ]:
mona_query = '''
with base as (
select  dt.taggable_id as retailer_id,
		dynamic_tags.name as dynamic_tag_name, 
		dynamic_tags.id as dynamic_tag_id, 
		c.id as new_cohort_id, 
		c.name as new_cohort_name
from dynamic_tags
inner join dynamic_taggables dt on dt.dynamic_tag_id = dynamic_tags.id
left join cohorts c on c.dynamic_tag_id = dynamic_tags.id
where dynamic_tags.name in ('mona_700', 'mona_701', 'mona_702', 'mona_703', 'mona_704', 'mona_1123', 'mona_1124', 'mona_1125', 'mona_1126', 'mona_cairo')), 

retailer_cohorts as (
select * 
from (
select distinct	
	taggable_id as retailer_id ,
	case when custom_ct =1 then cohort_id end as custom_cohort,
	case when custom_ct = 1 then fallbk else cohort_id end as main_cohort,
	61 as general_cohort, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from (
select 
	taggable_id,
	cohorts.id as cohort_id,
	FALLBACK_COHORT_ID as fallbk,
	cohorts.priority,
	case when coalesce(FALLBACK_COHORT_ID,61) = 61 then 0 else 1 end as custom_ct,
	rank () over (partition by taggable_id,custom_ct order by priority) as rk,
	sum (custom_ct) over (partition by taggable_id) as custom_count, 
	dynamic_tag_name, 
	new_cohort_id, 
	new_cohort_name
from cohorts
	left join dynamic_taggables as dt on dt.dynamic_tag_id  = cohorts.dynamic_tag_id
	inner join base on base.retailer_id = dt.taggable_id 
where cohorts.is_active = true and cohorts.id <> 61 
) as sub 
where rk = 1 and (custom_count = 0 or (custom_count >0 and custom_ct = 1))
order by 1)), 

served_warehouses as (
select  warehouse_id, 
		new_cohort_id, 
		count(distinct parent_sales_order_id) order_counts
from sales_orders 
inner join retailer_cohorts on retailer_cohorts.retailer_id = sales_orders.retailer_id 
where sales_orders.created_at::date between current_date - 90 and current_date 
group by all ), 

historical_stocks as (
select  distinct product_id, new_cohort_id
from materialized_views.stock_day_close std
inner join served_warehouses sw on sw.warehouse_id = std.warehouse_id 
where available_stock > 1
and timestamp::date between current_date - 30 and current_date), 

stock_now as (
select distinct product_id, new_cohort_id
from product_warehouse 
inner join served_warehouses sw on sw.warehouse_id = product_warehouse.warehouse_id 
where available_stock > 0), 

cohort_fallbacks as (
select  distinct custom_cohort, 
		main_cohort, 
		general_cohort, 
		new_cohort_id, 
		new_cohort_name
from retailer_cohorts), 

final as (
select  product_id as "Product ID", 
		product_name as "Product Name", 
		packing_unit_id as "Packing Unit ID", 
		price as "Price", 
		case when visibility = 1 then 'YES' else 'NO' end as "Visibility (YES/NO)", 
		null as "Execute At (format:dd/mm/yyyy HH:mm)", 
		null as "Tags", 
		new_cohort_id, 
		new_cohort_name
from (
select s.*,
	   coalesce(case when cppu3.is_customized = true then  cppu3.price end, cppu2.price,cppu.price) as price, 
	   case when c.section_id in (714,626, 516, 417, 351,121,285,20,54,37,10,43,44,36,14,8,55,619,622,621) and (stock_now.product_id is not null or historical_stocks.product_id is not null) then 1 else 0 end as visibility, 
	   p.name_ar as product_name
from (
select product_id, 
       packing_unit_id, 
	   custom_cohort, 
	   main_cohort, 
	   general_cohort, 
	   new_cohort_id, 
	   new_cohort_name, 
	   packing_unit_products.id as product_packing_unit_id
from packing_unit_products
cross join cohort_fallbacks
where DELETEd_AT is null
) s 
left join products p on p.id = s.product_id 
left join categories c on c.id = p.category_id 
left join cohort_product_packing_units as cppu on cppu.cohort_id = s.general_cohort and cppu.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu2 on cppu2.cohort_id = s.main_cohort and cppu2.product_packing_unit_id = s.product_packing_unit_id
left join cohort_product_packing_units as cppu3 on cppu3.cohort_id = s.custom_cohort and cppu3.product_packing_unit_id = s.product_packing_unit_id
left join historical_stocks on historical_stocks.product_id = s.product_id and historical_stocks.new_cohort_id = s.new_cohort_id
left join stock_now on stock_now.product_id = s.product_id and stock_now.new_cohort_id = s.new_cohort_id
)) 


select final.*,
       cppu.price AS db_price
from final
join packing_unit_products pup
  on pup.product_id = final."Product ID"
 and pup.packing_unit_id = final."Packing Unit ID"
left join cohort_product_packing_units as cppu
  on cppu.cohort_id = final.new_cohort_id
 and cppu.product_packing_unit_id = pup.id
where cppu.price <> final."Price"
'''
mona_data = query_snowflake(mona_query)
len(mona_data)

In [ ]:
for cohort_id in mona_data.NEW_COHORT_ID.unique():
    out = mona_data[mona_data['NEW_COHORT_ID'] == cohort_id]
    id_ = cohort_id
    name = out.NEW_COHORT_NAME.values[0]
    file_name_ = 'uploads/1_new_{}.xlsx'.format(name).replace(' ','_')
    out.to_excel(file_name_,index = False)
    ################### Loop to avoid file limit ######################
    # split file into chunks
    print('Spliting file into chunks...')
    if id_ == 61:
        chunks = [out[i:i + 2000] for i in range(0, len(out), 2000)]
    else:
        chunks = [out[i:i + 4000] for i in range(0, len(out), 4000)]
    print(f'len chunks = {len(chunks)}')
    fileslist = []
    for i, chunk in tqdm(enumerate(chunks), total=len(chunks)):
        fileslist.append(f'manual/output_{id_}_chunk_{i + 1}.xlsx')
        output_file_path = f'manual/output_{id_}_chunk_{i + 1}.xlsx'
        chunk.to_excel(output_file_path, index=False, engine='xlsxwriter')
    # Loop over chunks and upload
    print('Uploading...')
    for file in fileslist:
        chunk = file.split('chunk_')[1].split('.xls')[0]
        x = post_prices(id_, file)
        # print(str(x.content))
        if ('"success":true' in str(x.content).lower()):
            print(f"Prices are upoladed successfuly Region: {name} ,cohort: {id_}, chunk: {chunk}")
        else:
            print(f"ERROR Region: {name} ,cohort: {id_}, chunk: {chunk}")
            print(x.content)
            final_status = False
            break

In [10]:
# =============================================================================
# PUSH PRICES
# =============================================================================
MODE = 'live'  # Change to 'live' to actually push

df_ok = df_results[df_results['status'] == 'OK'].copy()

# Filter to only SKUs with stock
if len(df_ok) > 0:
    stock_by_product = df_stocks.groupby('product_id')['stocks'].sum().reset_index()
    df_ok = df_ok.merge(stock_by_product, on='product_id', how='left')
    no_stock = df_ok['stocks'].fillna(0) <= 0
    if no_stock.any():
        print(f'Skipping {no_stock.sum()} rows with no stock')
        df_ok = df_ok[~no_stock].copy()

if len(df_ok) == 0:
    print('No valid prices to push.')
else:
    wh_df = pd.DataFrame(WAREHOUSE_MAPPING, columns=['region', 'warehouse', 'warehouse_id', 'cohort_id'])

    push_rows = []
    for _, r in df_ok.iterrows():
        target_cohort = int(r['cohort_id'])
        cohort_whs = wh_df[wh_df['cohort_id'] == target_cohort]

        if cohort_whs.empty:
            cohort_whs = pd.DataFrame([{'warehouse_id': 1, 'cohort_id': target_cohort}])

        cur_price = r['current_price'] if pd.notna(r.get('current_price')) else 0

        for _, wh in cohort_whs.iterrows():
            push_rows.append({
                'product_id': int(r['product_id']),
                'sku': r['sku'],
                'new_price': r['new_price'],
                'warehouse_id': int(wh['warehouse_id']),
                'cohort_id': target_cohort,
                'stocks': 1,
                'current_price': cur_price,
            })

    df_push = pd.DataFrame(push_rows)

    n_products = df_ok['product_id'].nunique()
    n_cohorts = df_ok['cohort_id'].nunique()
    print(f'Pushing {n_products} products × {n_cohorts} cohorts = {len(df_push)} rows')
    print(f'Mode: {MODE}')
    print()

    result = push_prices(
        df_prices=df_push,
        pus=df_pus,
        source_module='manual_push',
        mode=MODE,
    )
    print(f'\nDone. Result: {result}')

Pushing 33 products × 9 cohorts = 50 rows
Mode: live


🚀 MODE: LIVE
   Files will be prepared AND uploaded to API
Loading disable_pu_visibility from Google Sheets...
  ✓ Loaded 88 products to disable min PU visibility

PUSH PRICES - Source: manual_push
Total received: 50
Price changes to push: 46
Skipped (no change): 4

Price changes summary:
  Increases: 45
  Decreases: 1

🔗 Mirrored prices to 6 main/general cohorts (+26 rows)
   Cohort 695 ← 700: 2 rows
   Cohort 61 ← 700: 2 rows
   Cohort 699 ← 702: 7 rows
   Cohort 697 ← 703: 3 rows
   Cohort 698 ← 704: 3 rows
   Cohort 696 ← 1123: 9 rows

📋 Prepared 68 packing unit prices (incl. main cohorts)

Processing cohort: 61
  Saved: uploads/manual_push_61.xlsx (2 rows)
  Split into 1 chunks (size: 2000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 216.86it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 695
  Saved: uploads/manual_push_695.xlsx (2 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 223.23it/s]

...

  Uploading...


    ✓ Chunk 1 uploaded successfully

Processing cohort: 1126
  Saved: uploads/manual_push_1126.xlsx (3 rows)
  Split into 1 chunks (size: 4000)


  Saving chunks: 100%|██████████| 1/1 [00:00<00:00, 226.00it/s]

  Uploading...


    ✓ Chunk 1 uploaded successfully

🚀 UPLOAD COMPLETE
Mode: live
Total prepared: 68
Total failed: 0
  Cleanup: removed 32 temporary .xlsx files from 2 folder(s)

Done. Result: {'total_received': 50, 'price_changes': 46, 'pushed': 68, 'failed': 0, 'skipped': 4, 'source_module': 'manual_push', 'timestamp': '2026-04-26 12:30:37', 'mode': 'live', 'skipped_cohorts': []}
